# Hampel fair value + Student-t GARCH(1,1)

**Goal:** robust fair value (causal Hampel) plus time-varying volatility from **GARCH(1,1) with Student-t innovations** to better handle fat tails than Gaussian GARCH.

- Loads **ASH_COATED_OSMIUM** from `prices_round_1_day_*.csv`.
- **Causal** Hampel (`center=False`): no look-ahead in FV/residuals.
- **Primary GARCH:** log returns × 100 → `arch_model(..., dist="t")` → conditional σ; scale back to log-return units.
- **Secondary GARCH:** first differences of Hampel residual (noise around FV).

Requires: `arch` (see project `pyproject.toml`). Lower `nu` in the fitted model ⇒ heavier tails.


In [22]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = Path(".")
files = [
    DATA_DIR / "prices_round_1_day_-2.csv",
    DATA_DIR / "prices_round_1_day_-1.csv",
    DATA_DIR / "prices_round_1_day_0.csv",
]
parts = []
for f in files:
    df = pd.read_csv(f, sep=";")
    m = re.search(r"day_(-?\d+)", f.name)
    if m:
        df["day"] = int(m.group(1))
    parts.append(df)
full = pd.concat(parts, ignore_index=True)
ash = full[full["product"] == "ASH_COATED_OSMIUM"].sort_values(["day", "timestamp"]).reset_index(drop=True)
ash["mid"] = pd.to_numeric(ash["mid_price"], errors="coerce").replace(0, np.nan)
ash = ash.dropna(subset=["mid"]).reset_index(drop=True)
print(f"ASH rows: {len(ash)}, days: {sorted(ash['day'].unique())}")
ash.head()


ASH rows: 29951, days: [np.int64(-2), np.int64(-1), np.int64(0)]


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,mid
0,-2,0,ASH_COATED_OSMIUM,NaN,NaN,NaN,NaN,NaN,NaN,10010.0,25.0,NaN,NaN,NaN,NaN,10010.0,0.0,10010.0
1,-2,100,ASH_COATED_OSMIUM,9992.0,15.0,NaN,NaN,NaN,NaN,10008.0,15.0,10011.0,20.0,NaN,NaN,10000.0,0.0,10000.0
2,-2,200,ASH_COATED_OSMIUM,9992.0,15.0,9989.0,30.0,NaN,NaN,10008.0,15.0,10010.0,30.0,NaN,NaN,10000.0,0.0,10000.0
3,-2,300,ASH_COATED_OSMIUM,9992.0,13.0,9989.0,26.0,NaN,NaN,10008.0,13.0,10010.0,26.0,NaN,NaN,10000.0,0.0,10000.0
4,-2,400,ASH_COATED_OSMIUM,9992.0,15.0,NaN,NaN,NaN,NaN,10008.0,15.0,10010.0,20.0,NaN,NaN,10000.0,0.0,10000.0


In [23]:
def causal_hampel_filter(series: pd.Series, window_size: int = 17, n_sigma: float = 3.0) -> pd.DataFrame:
    """Causal Hampel: only past/current data (center=False)."""
    x = series.astype(float)
    med = x.rolling(window=window_size, min_periods=window_size, center=False).median()
    mad = x.rolling(window=window_size, min_periods=window_size, center=False).apply(
        lambda a: np.median(np.abs(a - np.median(a))), raw=True
    )
    sigma = 1.4826 * mad
    thr = n_sigma * sigma
    raw_res = x - med
    storm = raw_res.abs() > thr
    fv = x.where(~storm, med)
    return pd.DataFrame(
        {
            "fair_value": fv,
            "rolling_median": med,
            "adaptive_threshold": thr,
            "residual": x - fv,
            "is_storm": storm,
        }
    )


W_HAMPEL = 17
N_SIGMA = 6.0
ham = causal_hampel_filter(ash["mid"], window_size=W_HAMPEL, n_sigma=N_SIGMA)
_ham_cols = ["fair_value", "rolling_median", "adaptive_threshold", "residual", "is_storm"]
ash = ash.drop(columns=[c for c in _ham_cols if c in ash.columns], errors="ignore")
ash = pd.concat([ash, ham], axis=1)
ash["log_ret"] = np.log(ash["mid"]).diff()
ash["resid_change"] = ash["residual"].diff()
print("Storm rate:", float(ash["is_storm"].mean()))
ash[["mid", "fair_value", "residual", "is_storm"]].describe()


Storm rate: 0.12697405762745817


,mid,fair_value,residual
count,29951.000000,29951.000000,29951.000000
mean,10000.204234,10000.205469,-0.001235
std,5.349695,5.009192,1.889413
min,9977.000000,9981.000000,-12.000000
25%,9997.000000,9997.000000,0.000000
50%,10000.500000,10000.500000,0.000000
75%,10003.500000,10003.000000,0.000000
max,10023.000000,10020.000000,13.000000


In [24]:
from arch import arch_model

# Student-t GARCH on log returns; let arch handle rescaling internally
y = ash["log_ret"].dropna()
print(f"Log-return GARCH sample size: {len(y)}")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    am = arch_model(
        y,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="t",
        rescale=True,
    )
    garch_ret = am.fit(disp="off")

print(garch_ret.summary())
nu_ret = float(garch_ret.params["nu"])
print(f"Fitted Student-t df (nu): {nu_ret:.3f}  (lower => fatter tails)")
print(f"Scale factor used by arch: {garch_ret.scale:.4f}")

ash = ash.join(
    pd.Series(
        garch_ret.conditional_volatility.values / garch_ret.scale,
        index=y.index,
        name="garch_cond_vol_logret",
    ),
    how="left",
)

ROLL = 50
ash["roll_vol_logret"] = ash["log_ret"].rolling(ROLL, min_periods=ROLL).std()


Log-return GARCH sample size: 29950
                        Constant Mean - GARCH Model Results                         
Dep. Variable:                      log_ret   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -3542.60
Distribution:      Standardized Student's t   AIC:                           7095.20
Method:                  Maximum Likelihood   BIC:                           7136.73
                                              No. Observations:                29950
Date:                      Thu, Apr 16 2026   Df Residuals:                    29949
Time:                              11:28:54   Df Model:                            1
                                  Mean Model                                 
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-------------------------------------------

In [25]:
# Student-t GARCH on Hampel residual changes (fat-tailed noise around FV)
z = ash["resid_change"].dropna().replace([np.inf, -np.inf], np.nan).dropna()
print(f"Residual-change GARCH sample size: {len(z)}")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    am2 = arch_model(z, mean="Constant", vol="GARCH", p=1, q=1, dist="t", rescale=True)
    garch_resid = am2.fit(disp="off")

print(garch_resid.summary())
nu_z = float(garch_resid.params["nu"])
print(f"Residual noise nu: {nu_z:.3f}")
print(f"Scale factor used by arch: {garch_resid.scale:.4f}")

ash = ash.join(
    pd.Series(
        garch_resid.conditional_volatility.values / garch_resid.scale,
        index=z.index,
        name="garch_cond_vol_resid_d1",
    ),
    how="left",
)


Residual-change GARCH sample size: 29950
                        Constant Mean - GARCH Model Results                         
Dep. Variable:                 resid_change   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -44135.1
Distribution:      Standardized Student's t   AIC:                           88280.3
Method:                  Maximum Likelihood   BIC:                           88321.8
                                              No. Observations:                29950
Date:                      Thu, Apr 16 2026   Df Residuals:                    29949
Time:                              11:28:54   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------

In [26]:
idx = np.arange(len(ash))
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.32, 0.22, 0.23, 0.23],
    subplot_titles=(
        "Mid vs Hampel FV",
        "Student-t GARCH σ (log returns) vs rolling std",
        "Student-t GARCH σ (Δ residual)",
        "Hampel storm flag",
    ),
)
fig.add_trace(go.Scatter(x=idx, y=ash["mid"], name="mid", line=dict(color="lightgray", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=idx, y=ash["fair_value"], name="Hampel FV", line=dict(color="blue", width=1)), row=1, col=1)
fig.add_trace(
    go.Scatter(x=idx, y=ash["garch_cond_vol_logret"], name="GARCH σ (log ret)", line=dict(color="crimson", width=1)),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=idx,
        y=ash["roll_vol_logret"],
        name=f"Roll std {ROLL}",
        line=dict(color="gray", width=1, dash="dot"),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=idx,
        y=ash["garch_cond_vol_resid_d1"],
        name="GARCH σ (Δ resid)",
        line=dict(color="darkgreen", width=1),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(x=idx, y=ash["is_storm"].astype(float), name="storm", line=dict(color="orange", width=0.8)),
    row=4,
    col=1,
)
fig.update_layout(
    height=1000,
    template="plotly_white",
    title_text="ASH: causal Hampel + Student-t GARCH(1,1)",
)
fig.show()


In [27]:
# Residual diagnostics parameter search (Hampel + Student-t GARCH)
from itertools import product
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model


def residual_diagnostics(x: pd.Series, lb_lag: int = 10, arch_lag: int = 10):
    """Return whiteness/heteroskedasticity diagnostics on a series."""
    x = x.dropna()
    if len(x) < max(lb_lag + 5, arch_lag + 5):
        return {
            "lb_p": np.nan,
            "lb2_p": np.nan,
            "arch_p": np.nan,
            "kurt": np.nan,
        }
    lb_p = float(acorr_ljungbox(x, lags=[lb_lag], return_df=True)["lb_pvalue"].iloc[0])
    lb2_p = float(acorr_ljungbox(x**2, lags=[lb_lag], return_df=True)["lb_pvalue"].iloc[0])
    _, arch_p, _, _ = het_arch(x, nlags=arch_lag)
    return {
        "lb_p": lb_p,
        "lb2_p": float(arch_p) if np.isfinite(arch_p) else np.nan,
        "arch_p": float(arch_p) if np.isfinite(arch_p) else np.nan,
        "kurt": float(pd.Series(x).kurtosis()),
    }


# Candidate grids
W_GRID = [9, 11, 13, 15, 17, 21]
N_SIGMA_GRID = [2.5, 3.0, 3.5]
PQ_GRID = [(1, 1), (1, 2), (2, 1), (2, 2)]
LB_LAG = 10
ARCH_LAG = 10

rows = []

for w, n_sig in product(W_GRID, N_SIGMA_GRID):
    ham_tmp = causal_hampel_filter(ash["mid"], window_size=w, n_sigma=n_sig)
    resid = ham_tmp["residual"].dropna()

    if len(resid) < 200:
        continue

    base_diag = residual_diagnostics(resid, lb_lag=LB_LAG, arch_lag=ARCH_LAG)
    storm_rate = float(ham_tmp["is_storm"].mean())

    # GARCH on log returns, Student-t innovations
    y = ash["log_ret"].dropna()
    for p, q in PQ_GRID:
        try:
            am = arch_model(y, mean="Constant", vol="GARCH", p=p, q=q, dist="t", rescale=True)
            res = am.fit(disp="off")
            nu = float(res.params.get("nu", np.nan))
            aic = float(res.aic)
            bic = float(res.bic)

            std_resid = pd.Series(res.std_resid).replace([np.inf, -np.inf], np.nan).dropna()
            g_diag = residual_diagnostics(std_resid, lb_lag=LB_LAG, arch_lag=ARCH_LAG)

            # Composite score: lower is better
            # Encourage white-noise residuals (high p-values), lower information criteria, plausible storm rate.
            penalty_whiteness = (1 - np.nan_to_num(base_diag["lb_p"])) + (1 - np.nan_to_num(g_diag["lb_p"]))
            penalty_arch = (1 - np.nan_to_num(g_diag["arch_p"]))
            penalty_storm = abs(storm_rate - 0.18)  # calibrated target from previous notebook results
            score = (
                0.6 * penalty_whiteness
                + 0.3 * penalty_arch
                + 0.1 * penalty_storm
                + 0.00001 * (aic - y.shape[0])  # scale AIC contribution
            )

            rows.append(
                {
                    "W": w,
                    "n_sigma": n_sig,
                    "p": p,
                    "q": q,
                    "storm_rate": storm_rate,
                    "hampel_lb_p": base_diag["lb_p"],
                    "hampel_arch_p": base_diag["arch_p"],
                    "garch_aic": aic,
                    "garch_bic": bic,
                    "nu": nu,
                    "stdres_lb_p": g_diag["lb_p"],
                    "stdres_arch_p": g_diag["arch_p"],
                    "score": score,
                }
            )
        except Exception:
            continue

search_df = pd.DataFrame(rows).sort_values(["score", "garch_aic"], ascending=[True, True]).reset_index(drop=True)
print(f"Parameter combinations evaluated: {len(search_df)}")
display(search_df.head(15))


Parameter combinations evaluated: 72


,W,n_sigma,p,q,storm_rate,hampel_lb_p,hampel_arch_p,garch_aic,garch_bic,nu,stdres_lb_p,stdres_arch_p,score
0,15,3.0,1,1,0.197222,0.725958,0.310756,7095.195718,7136.732141,2.662871,0.0,3.500666e-19,0.837599
1,15,3.0,2,1,0.197222,0.725958,0.310756,7097.195719,7147.039427,2.662887,0.0,3.497559e-19,0.837619
2,15,3.0,1,2,0.197222,0.725958,0.310756,7097.195719,7147.039427,2.662870,0.0,3.500515e-19,0.837619
3,15,3.0,2,2,0.197222,0.725958,0.310756,7099.195865,7157.346857,2.662871,0.0,3.499335e-19,0.837639
4,17,3.5,1,1,0.172682,0.716463,0.464896,7095.195718,7136.732141,2.662871,0.0,3.500666e-19,0.842306
5,17,3.5,2,1,0.172682,0.716463,0.464896,7097.195719,7147.039427,2.662887,0.0,3.497559e-19,0.842326
6,17,3.5,1,2,0.172682,0.716463,0.464896,7097.195719,7147.039427,2.662870,0.0,3.500515e-19,0.842326
7,17,3.5,2,2,0.172682,0.716463,0.464896,7099.195865,7157.346857,2.662871,0.0,3.499335e-19,0.842346
8,13,3.5,1,1,0.193650,0.705076,0.240061,7095.195718,7136.732141,2.662871,0.0,3.500666e-19,0.849771
9,13,3.5,2,1,0.193650,0.705076,0.240061,7097.195719,7147.039427,2.662887,0.0,3.497559e-19,0.849791


In [28]:
# Select best parameters and refit final model
best = search_df.iloc[0].to_dict()
print("Best combo from residual diagnostics:")
for k in ["W", "n_sigma", "p", "q", "storm_rate", "nu", "hampel_lb_p", "stdres_lb_p", "stdres_arch_p", "garch_aic", "score"]:
    print(f"  {k}: {best[k]}")

W_BEST = int(best["W"])
NSIG_BEST = float(best["n_sigma"])
P_BEST = int(best["p"])
Q_BEST = int(best["q"])

# Recompute Hampel with best params
ham_best = causal_hampel_filter(ash["mid"], window_size=W_BEST, n_sigma=NSIG_BEST)
ash_best = ash.copy()
ash_best[["fair_value_best", "rolling_median_best", "adaptive_threshold_best", "residual_best", "is_storm_best"]] = ham_best[
    ["fair_value", "rolling_median", "adaptive_threshold", "residual", "is_storm"]
]

# Refit Student-t GARCH with best (p,q)
y = ash_best["log_ret"].dropna()
best_am = arch_model(y, mean="Constant", vol="GARCH", p=P_BEST, q=Q_BEST, dist="t", rescale=True)
best_res = best_am.fit(disp="off")
print("\nFinal best-fit GARCH summary:")
print(best_res.summary())
print(f"Scale factor used by arch: {best_res.scale:.4f}")

ash_best = ash_best.join(
    pd.Series(best_res.conditional_volatility.values / best_res.scale, index=y.index, name="garch_cond_vol_best"),
    how="left",
)

# Compact diagnostics report
resid_diag_best = residual_diagnostics(ash_best["residual_best"].dropna(), lb_lag=10, arch_lag=10)
stdres_diag_best = residual_diagnostics(pd.Series(best_res.std_resid).dropna(), lb_lag=10, arch_lag=10)

report = pd.DataFrame(
    [
        {"metric": "Hampel residual LB p(10)", "value": resid_diag_best["lb_p"]},
        {"metric": "Hampel residual ARCH p(10)", "value": resid_diag_best["arch_p"]},
        {"metric": "Std resid LB p(10)", "value": stdres_diag_best["lb_p"]},
        {"metric": "Std resid ARCH p(10)", "value": stdres_diag_best["arch_p"]},
        {"metric": "Student-t nu", "value": float(best_res.params.get("nu", np.nan))},
        {"metric": "Storm rate", "value": float(ash_best["is_storm_best"].mean())},
        {"metric": "AIC", "value": float(best_res.aic)},
        {"metric": "BIC", "value": float(best_res.bic)},
    ]
)
display(report)


Best combo from residual diagnostics:
  W: 15.0
  n_sigma: 3.0
  p: 1.0
  q: 1.0
  storm_rate: 0.19722212947814763
  nu: 2.662871108726425
  hampel_lb_p: 0.7259584776033895
  stdres_lb_p: 0.0
  stdres_arch_p: 3.5006655864176357e-19
  garch_aic: 7095.195718224577
  score: 0.8375990835680267

Final best-fit GARCH summary:
                        Constant Mean - GARCH Model Results                         
Dep. Variable:                      log_ret   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -3542.60
Distribution:      Standardized Student's t   AIC:                           7095.20
Method:                  Maximum Likelihood   BIC:                           7136.73
                                              No. Observations:                29950
Date:                      Thu, Apr 16 2026   Df Residuals:                    2994

,metric,value
0,Hampel residual LB p(10),7.259585e-01
1,Hampel residual ARCH p(10),3.107561e-01
2,Std resid LB p(10),0.000000e+00
3,Std resid ARCH p(10),3.500666e-19
4,Student-t nu,2.662871e+00
5,Storm rate,1.972221e-01
6,AIC,7.095196e+03
7,BIC,7.136732e+03


## Robust FV Comparison: Hampel vs Hampel+Kalman vs Hampel+Huber

This section tests fair-value (FV) estimators on the same data and split:

- **Hampel-only FV** (causal robust outlier handling)
- **Hampel + local-trend Kalman** (state-space smoothing)
- **Hampel + rolling Huber regression** (robust local trend)

Protocol:

1. Reuse best Hampel parameters from `search_df` when available.
2. Train Kalman noise parameters (`R`, `Q_level`, `Q_drift`) on train days only.
3. Evaluate all models on holdout day (default: earliest day, usually -2).
4. Report multi-horizon MAE/RMSE and residual diagnostics (Ljung-Box, ARCH).


In [30]:
import math

from scipy.optimize import minimize
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

try:
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import HuberRegressor
except ImportError as e:
    raise ImportError("scikit-learn is required for Huber test. Install it in your venv.") from e


def _kalman_filter_local_trend(z: pd.Series, r_obs: float, q_level: float, q_drift: float) -> pd.Series:
    """Causal local-trend Kalman filter: state=[level, drift], obs=level."""
    x = np.asarray(z, dtype=float)
    n = len(x)
    out = np.full(n, np.nan, dtype=float)

    finite_idx = np.where(np.isfinite(x))[0]
    if finite_idx.size == 0:
        return pd.Series(out, index=z.index, name="fv_kalman")

    mu = float(x[finite_idx[0]])
    beta = 0.0
    P = np.array([[25.0, 0.0], [0.0, 4.0]], dtype=float)

    F = np.array([[1.0, 1.0], [0.0, 1.0]], dtype=float)
    Q = np.array([[q_level, 0.0], [0.0, q_drift]], dtype=float)
    H = np.array([[1.0, 0.0]], dtype=float)
    I = np.eye(2, dtype=float)

    for t in range(n):
        x_pred = F @ np.array([mu, beta], dtype=float)
        P_pred = F @ P @ F.T + Q

        if np.isfinite(x[t]):
            innov = float(x[t] - np.squeeze(H @ x_pred))
            S = float(np.squeeze(H @ P_pred @ H.T) + r_obs)
            S = max(S, 1e-12)
            K = (P_pred @ H.T).flatten() / S
            x_new = x_pred + K * innov
            P = (I - np.outer(K, H)) @ P_pred
            mu, beta = float(x_new[0]), float(x_new[1])
        else:
            mu, beta = float(x_pred[0]), float(x_pred[1])
            P = P_pred

        out[t] = mu

    return pd.Series(out, index=z.index, name="fv_kalman")


def _kalman_innovation_loglike(z: np.ndarray, r_obs: float, q_level: float, q_drift: float) -> float:
    z = np.asarray(z, dtype=float)
    z = z[np.isfinite(z)]
    if len(z) < 10:
        return -1e18

    mu = float(z[0])
    beta = 0.0
    P = np.array([[25.0, 0.0], [0.0, 4.0]], dtype=float)

    F = np.array([[1.0, 1.0], [0.0, 1.0]], dtype=float)
    Q = np.array([[q_level, 0.0], [0.0, q_drift]], dtype=float)
    H = np.array([[1.0, 0.0]], dtype=float)
    I = np.eye(2, dtype=float)

    ll = 0.0
    for t in range(1, len(z)):
        x_pred = F @ np.array([mu, beta], dtype=float)
        P_pred = F @ P @ F.T + Q

        innov = float(z[t] - np.squeeze(H @ x_pred))
        S = float(np.squeeze(H @ P_pred @ H.T) + r_obs)
        S = max(S, 1e-12)

        K = (P_pred @ H.T).flatten() / S
        x_new = x_pred + K * innov
        P = (I - np.outer(K, H)) @ P_pred
        mu, beta = float(x_new[0]), float(x_new[1])

        ll += -0.5 * math.log(2.0 * math.pi * S) - 0.5 * (innov * innov) / S

    return float(ll)


def fit_kalman_mle(train_series: pd.Series) -> dict:
    """Fit Kalman noise params on train data only."""
    z = np.asarray(train_series, dtype=float)

    log_bounds = [
        (math.log(1e-4), math.log(1e4)),   # R
        (math.log(1e-8), math.log(1e2)),   # Q_level
        (math.log(1e-10), math.log(1e0)),  # Q_drift
    ]

    def objective(u: np.ndarray) -> float:
        r = math.exp(float(u[0]))
        ql = math.exp(float(u[1]))
        qd = math.exp(float(u[2]))
        ll = _kalman_innovation_loglike(z, r, ql, qd)
        return 1e12 if not np.isfinite(ll) else -ll

    x0 = np.array([math.log(1.0), math.log(0.3), math.log(1e-3)], dtype=float)
    res = minimize(
        objective,
        x0,
        method="L-BFGS-B",
        bounds=log_bounds,
        options={"maxiter": 120, "ftol": 1e-8},
    )

    return {
        "R": float(math.exp(res.x[0])),
        "Q_level": float(math.exp(res.x[1])),
        "Q_drift": float(math.exp(res.x[2])),
        "success": bool(res.success),
        "message": str(res.message),
    }


def rolling_huber_fv(series: pd.Series, window: int = 40, epsilon: float = 1.35) -> pd.Series:
    """Causal rolling Huber trend estimate with robust scaling and safe fallbacks."""
    x = pd.Series(series).astype(float)
    arr = x.values
    n = len(arr)
    out = np.full(n, np.nan, dtype=float)

    for t in range(n):
        lo = max(0, t - window + 1)
        y_win = arr[lo : t + 1]
        ok = np.isfinite(y_win)
        if ok.sum() < 8:
            continue

        y = y_win[ok].astype(float)
        t_idx = np.arange(len(y_win), dtype=float)[ok]

        # Robust centering/scaling reduces Huber optimizer failures.
        x_mu = float(np.mean(t_idx))
        x_sd = float(np.std(t_idx))
        if not np.isfinite(x_sd) or x_sd < 1e-8:
            x_sd = 1.0
        y_mu = float(np.median(y))
        y_mad = float(np.median(np.abs(y - y_mu)))
        y_sd = 1.4826 * y_mad
        if not np.isfinite(y_sd) or y_sd < 1e-8:
            y_sd = float(np.std(y)) if np.isfinite(np.std(y)) and np.std(y) > 1e-8 else 1.0

        Xs = ((t_idx - x_mu) / x_sd).reshape(-1, 1)
        ys = (y - y_mu) / y_sd
        x_pred = (float(len(y_win) - 1) - x_mu) / x_sd

        try:
            mdl = HuberRegressor(epsilon=epsilon, alpha=1e-6, max_iter=500, tol=1e-5)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                mdl.fit(Xs, ys)
            yhat_s = float(mdl.predict(np.array([[x_pred]], dtype=float))[0])
            out[t] = y_mu + y_sd * yhat_s
        except Exception:
            # Fallback: simple least-squares line on standardized data.
            try:
                b1, b0 = np.polyfit(Xs.ravel(), ys, 1)
                out[t] = y_mu + y_sd * (b1 * x_pred + b0)
            except Exception:
                out[t] = y_mu

    return pd.Series(out, index=x.index, name="fv_huber")


def residual_diag(x: pd.Series, lb_lag: int = 10, arch_lag: int = 10) -> dict:
    s = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < max(lb_lag, arch_lag) + 5:
        return {"lb_p": np.nan, "arch_p": np.nan, "n": len(s)}
    lb = acorr_ljungbox(s, lags=[lb_lag], return_df=True)
    arch_stat, arch_p, _, _ = het_arch(s, nlags=arch_lag)
    return {"lb_p": float(lb["lb_pvalue"].iloc[0]), "arch_p": float(arch_p), "n": len(s)}


def evaluate_fv(df: pd.DataFrame, fv_col: str, horizons=(1, 3, 5, 10), eval_mask=None) -> pd.DataFrame:
    rows = []
    base_mask = pd.Series(True, index=df.index) if eval_mask is None else pd.Series(eval_mask, index=df.index)
    for h in horizons:
        fut = df["mid"].shift(-h)
        e = fut - df[fv_col]
        ok = base_mask & np.isfinite(df[fv_col]) & np.isfinite(fut)
        if ok.sum() == 0:
            rows.append({"model": fv_col, "h": h, "n": 0, "mae": np.nan, "rmse": np.nan, "bias": np.nan})
            continue
        ee = e[ok]
        rows.append(
            {
                "model": fv_col,
                "h": h,
                "n": int(ok.sum()),
                "mae": float(np.mean(np.abs(ee))),
                "rmse": float(np.sqrt(np.mean(np.square(ee)))),
                "bias": float(np.mean(ee)),
            }
        )
    return pd.DataFrame(rows)


In [31]:
# --- Build comparable FV models ---
# Reuse best Hampel params from search if available
if "search_df" in globals() and isinstance(search_df, pd.DataFrame) and len(search_df) > 0:
    _best = search_df.iloc[0]
    W_FV = int(_best["W"])
    NS_FV = float(_best["n_sigma"])
else:
    W_FV = 15
    NS_FV = 3.0

print(f"Using Hampel params: W={W_FV}, n_sigma={NS_FV}")

ham_fv = causal_hampel_filter(ash["mid"], window_size=W_FV, n_sigma=NS_FV)
ash_eval = ash[["day", "timestamp", "mid"]].copy()
ash_eval["fv_hampel"] = ham_fv["fair_value"]

# Train/test split by day: hold out earliest day (usually -2)
days = sorted(pd.Series(ash_eval["day"]).dropna().unique().tolist())
if len(days) < 2:
    raise ValueError("Need at least 2 days for walk-forward holdout.")

holdout_day = days[0]
train_mask = ash_eval["day"] != holdout_day
test_mask = ash_eval["day"] == holdout_day
print(f"Train days: {[d for d in days if d != holdout_day]} | Holdout day: {holdout_day}")

# Kalman fitted on train portion of Hampel-cleaned input
kalman_params = fit_kalman_mle(ash_eval.loc[train_mask, "fv_hampel"])
print("Kalman MLE params:")
for k in ["R", "Q_level", "Q_drift", "success", "message"]:
    print(f"  {k}: {kalman_params[k]}")

ash_eval["fv_kalman"] = _kalman_filter_local_trend(
    ash_eval["fv_hampel"],
    r_obs=kalman_params["R"],
    q_level=kalman_params["Q_level"],
    q_drift=kalman_params["Q_drift"],
)

# Huber on Hampel-cleaned series
HUBER_WINDOW = 40
HUBER_EPS = 1.35
print(f"Running rolling Huber (window={HUBER_WINDOW}, epsilon={HUBER_EPS}) ...")
ash_eval["fv_huber"] = rolling_huber_fv(ash_eval["fv_hampel"], window=HUBER_WINDOW, epsilon=HUBER_EPS)

# --- Evaluate on holdout only ---
HORIZONS = (1, 3, 5, 10)
res_tbl = pd.concat(
    [
        evaluate_fv(ash_eval, "fv_hampel", horizons=HORIZONS, eval_mask=test_mask),
        evaluate_fv(ash_eval, "fv_kalman", horizons=HORIZONS, eval_mask=test_mask),
        evaluate_fv(ash_eval, "fv_huber", horizons=HORIZONS, eval_mask=test_mask),
    ],
    ignore_index=True,
)
print("\nHoldout forecasting error table (lower MAE/RMSE is better):")
display(res_tbl.sort_values(["h", "rmse"]).reset_index(drop=True))

# Residual diagnostics on holdout
diag_rows = []
for col in ["fv_hampel", "fv_kalman", "fv_huber"]:
    resid = ash_eval.loc[test_mask, "mid"] - ash_eval.loc[test_mask, col]
    d = residual_diag(resid, lb_lag=10, arch_lag=10)
    diag_rows.append({"model": col, **d})

diag_df = pd.DataFrame(diag_rows)
print("\nHoldout residual diagnostics (higher p-values are better):")
display(diag_df)

# Quick visual on holdout day
hold = ash_eval.loc[test_mask].copy()
fig_cmp = make_subplots(rows=1, cols=1)
fig_cmp.add_trace(go.Scatter(x=hold["timestamp"], y=hold["mid"], name="mid", line=dict(color="lightgray", width=1)))
fig_cmp.add_trace(go.Scatter(x=hold["timestamp"], y=hold["fv_hampel"], name="fv_hampel", line=dict(color="royalblue", width=1)))
fig_cmp.add_trace(go.Scatter(x=hold["timestamp"], y=hold["fv_kalman"], name="fv_kalman", line=dict(color="crimson", width=1)))
fig_cmp.add_trace(go.Scatter(x=hold["timestamp"], y=hold["fv_huber"], name="fv_huber", line=dict(color="darkgreen", width=1)))
fig_cmp.update_layout(
    template="plotly_white",
    title=f"Holdout day {holdout_day}: Mid vs FV estimators",
    height=450,
)
fig_cmp.show()


Using Hampel params: W=15, n_sigma=3.0
Train days: [-1, 0] | Holdout day: -2
Kalman MLE params:
  R: 0.876799740830517
  Q_level: 0.08292008764959251
  Q_drift: 9.999999999999996e-11
  success: True
  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Running rolling Huber (window=40, epsilon=1.35) ...


c:\Users\alexn\Documents\dev\projects\imc_prosperity_tut\.venv\Lib\site-packages\sklearn\linear_model\_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
c:\Users\alexn\Documents\dev\projects\imc_prosperity_tut\.venv\Lib\site-packages\sklearn\linear_model\_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, sel

ValueError: HuberRegressor convergence failed: l-BFGS-b solver terminated with ABNORMAL: 